In [7]:
import xarray as xr
import numpy as np

# --- Open SWC and SIF datasets properly ---
swc_ds = xr.open_dataset("gs_drought_swc_8daymin_2001_2022.nc")

# After you know the variable name (e.g. 'swc', 'SWC_F_MDS_1', etc.)
swc = swc_ds["swc"]  # <-- replace with the exact variable name printed above

sif_ds = xr.open_dataset("lai_anoma.nc")
sif = sif_ds["lai"]  # <-- replace with correct variable name

#sif_ds = xr.open_dataset("SIFratio.nc")
#sif = sif_ds["SIFrel"]  # <-- replace with correct variable name

# --- Open threshold dataset and select variable ---
thr_ds = xr.open_dataset("threshold_metrics_8daymin_gosif_gpp.nc")
thr = thr_ds["thr_final_second"]/100


In [8]:
# --- broadcast threshold to (time, lat, lon) ---
thr_b = thr.broadcast_like(swc)

# --- masks ---
mask_below = swc <  thr_b
mask_above = swc >= thr_b

# --- masked series ---
x_b = swc.where(mask_below)
y_b = sif.where(mask_below)
x_a = swc.where(mask_above)
y_a = sif.where(mask_above)

In [9]:
def fast_slope(x, y, dim="time"):
    # β = Cov(x,y)/Var(x), using NaN-safe reductions
    xm = x.mean(dim=dim, skipna=True)
    ym = y.mean(dim=dim, skipna=True)
    cov = ((x - xm) * (y - ym)).sum(dim=dim, skipna=True)
    varx = ((x - xm) ** 2).sum(dim=dim, skipna=True)
    beta = cov / varx
    return beta.where(varx > 0)

# --- sensitivities ---
sens_below = fast_slope(x_b, y_b, dim="time")
sens_above = fast_slope(x_a, y_a, dim="time")

In [10]:
sens_below.astype("float32").to_netcdf("sensitivity_lai_to_swc_belowThr.nc")
sens_above.astype("float32").to_netcdf("sensitivity_lai_to_swc_aboveThr.nc")

In [6]:
sens_below.astype("float32").to_netcdf("sensitivity_sifratio_to_swc_belowThr.nc")
sens_above.astype("float32").to_netcdf("sensitivity_sifratio_to_swc_aboveThr.nc")